# Genomic Selection Tutorial - Clonal Breeding

This notebook replicates the AlphaSimR clonal breeding genomic selection tutorial using AlphaSimPy.
It demonstrates genomic selection in a clonal tea breeding program with multiple evaluation stages.

**Authors**: Translated from AlphaSimR tutorial by Nelson Lubanga, Gregor Gorjanc, Jon Bancic, Philip Greenspoon, Chris Gaynor  
**Date**: 2024  
**Package**: AlphaSimPy

This tutorial applies GS to replace early-stage phenotypic evaluation (HPT1-3) with genomic prediction, 
accelerating the breeding cycle while maintaining genetic gain.

## Import Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve
from AlphaSimPy import (
    runMacs2, SimParam, newPop, randCross, setPheno, selectInd,
    meanG, varG, mergePops
)

print("AlphaSimPy Clonal Breeding - Genomic Selection Tutorial")
print("All libraries imported successfully!")

## Helper Functions for Genomic Selection

These functions implement RRBLUP and setEBV functionality for genomic selection, plus a helper for subsetting Pop objects.

In [ ]:
def subsetPop(pop, indices):
    """
    Subset a Pop object by indices (helper function since Pop doesn't support indexing).
    
    Parameters:
    -----------
    pop : Pop
        Population object
    indices : list or slice
        Indices to select
    
    Returns:
    --------
    Pop
        Subsetted population
    """
    from AlphaSimPy import Pop
    
    if isinstance(indices, slice):
        indices = list(range(*indices.indices(pop.n_ind)))
    
    if not indices:
        # Return empty population
        return Pop(
            n_ind=0, n_chr=pop.n_chr, ploidy=pop.ploidy, n_loci=pop.n_loci,
            geno=[], gen_map=pop.gen_map, centromere=pop.centromere,
            inbred=pop.inbred, id=[], iid=[], mother=[], father=[],
            sex=[], n_traits=pop.n_traits, gv=np.empty((0, pop.n_traits)),
            pheno=np.empty((0, pop.n_traits)), ebv=np.empty((0, 0)),
            gxe=pop.gxe, fix_eff=[], misc={}, misc_pop={}
        )
    
    return Pop(
        n_ind=len(indices), n_chr=pop.n_chr, ploidy=pop.ploidy, n_loci=pop.n_loci,
        geno=[pop.geno[chr_idx][:, :, indices] for chr_idx in range(pop.n_chr)],
        gen_map=pop.gen_map, centromere=pop.centromere, inbred=pop.inbred,
        id=[pop.id[i] for i in indices],
        iid=[pop.iid[i] for i in indices],
        mother=[pop.mother[i] for i in indices],
        father=[pop.father[i] for i in indices],
        sex=[pop.sex[i] for i in indices],
        n_traits=pop.n_traits, gv=pop.gv[indices, :],
        pheno=pop.pheno[indices, :], ebv=pop.ebv[indices, :],
        gxe=pop.gxe, fix_eff=[pop.fix_eff[i] for i in indices],
        misc=pop.misc, misc_pop=pop.misc_pop
    )


def pullSnpGeno(pop, simParam, snpChip=1):
    """
    Extract SNP genotype matrix from a population.
    
    Parameters:
    -----------
    pop : Pop
        Population object
    simParam : SimParam
        Simulation parameters
    snpChip : int
        Which SNP chip to use (1-indexed)
    
    Returns:
    --------
    np.ndarray
        Matrix of SNP genotypes (n_ind x n_snp)
    """
    if snpChip < 1 or snpChip > len(simParam.snp_chips):
        raise ValueError(f"snpChip {snpChip} not available")
    
    snp_chip = simParam.snp_chips[snpChip - 1]
    n_snp = sum(snp_chip.loci_per_chr)
    
    if n_snp == 0:
        raise ValueError("No SNPs available in specified chip")
    
    # Extract genotypes for SNP loci
    geno_matrix = np.zeros((pop.n_ind, n_snp), dtype=np.float64)
    
    snp_idx = 0
    for chr_idx in range(pop.n_chr):
        n_snp_chr = snp_chip.loci_per_chr[chr_idx]
        if n_snp_chr == 0:
            continue
        
        # Get SNP locations on this chromosome
        snp_loci = snp_chip.loci_loc[snp_idx:snp_idx + n_snp_chr]
        
        # Extract genotypes from packed format
        for ind_idx in range(pop.n_ind):
            for snp_loc_idx, snp_loc in enumerate(snp_loci):
                # Extract genotype from packed byte format
                byte_idx = snp_loc // 8
                bit_idx = snp_loc % 8
                
                # Sum across ploidy
                genotype = 0
                for p in range(pop.ploidy):
                    byte_val = pop.geno[chr_idx][byte_idx, p, ind_idx]
                    bit_val = (byte_val >> bit_idx) & 1
                    genotype += bit_val
                
                geno_matrix[ind_idx, snp_idx + snp_loc_idx] = genotype
        
        snp_idx += n_snp_chr
    
    return geno_matrix


def RRBLUP(trainPop, simParam, traits=1, use="pheno", snpChip=1, useReps=False):
    """
    Fit an RR-BLUP model for genomic predictions.
    
    Parameters:
    -----------
    trainPop : Pop
        Training population
    simParam : SimParam
        Simulation parameters
    traits : int
        Trait to model (1-indexed)
    use : str
        Use "pheno", "gv", or "ebv" for training
    snpChip : int
        Which SNP chip to use
    useReps : bool
        Whether to account for replication in training (simplified implementation)
    
    Returns:
    --------
    dict
        Dictionary containing model coefficients and metadata
    """
    # Get response variable
    if use == "pheno":
        y = trainPop.pheno[:, traits - 1]
    elif use == "gv":
        y = trainPop.gv[:, traits - 1]
    elif use == "ebv":
        y = trainPop.ebv[:, traits - 1]
    else:
        raise ValueError(f"use='{use}' is not a valid option")
    
    # Remove missing values
    valid_idx = ~np.isnan(y)
    y = y[valid_idx]
    
    if len(y) == 0:
        raise ValueError("No valid observations for training")
    
    # Get SNP genotypes
    M = pullSnpGeno(trainPop, simParam, snpChip)
    M = M[valid_idx, :]
    
    # Center genotypes
    M_mean = np.mean(M, axis=0)
    M_centered = M - M_mean
    
    # Fit RR-BLUP using mixed model equations
    # y = Xb + Zu + e
    # where Z is the centered marker matrix
    # We use the GBLUP equivalent: K = ZZ'/p where p is number of markers
    
    n_markers = M_centered.shape[1]
    if n_markers == 0:
        raise ValueError("No markers available")
    
    # Calculate genomic relationship matrix G = ZZ' / p
    G = np.dot(M_centered, M_centered.T) / n_markers
    
    # Add small value to diagonal for numerical stability
    G += np.eye(G.shape[0]) * 1e-6
    
    # Estimate variance components (simplified - using fixed lambda)
    # In practice, you would estimate these, but for tutorial we use fixed values
    # If useReps=True, we could adjust lambda based on replication, but simplified here
    lambda_val = n_markers / 10.0  # Simplified lambda
    
    # Solve for BLUP: (G + lambda*I) * u = y
    # where u are the breeding values
    A = G + lambda_val * np.eye(G.shape[0])
    u = solve(A, y, assume_a='pos')
    
    # Store model for prediction
    model = {
        'u': u,  # BLUP solutions
        'M_mean': M_mean,  # Mean marker values for centering
        'M_train': M_centered,  # Training marker matrix (centered)
        'y_train': y,  # Training phenotypes
        'lambda': lambda_val,  # Regularization parameter
        'n_markers': n_markers,
        'trait': traits,
        'valid_idx': valid_idx
    }
    
    return model


def setEBV(pop, gsModel, simParam, snpChip=1):
    """
    Set estimated breeding values (EBV) for a population using a genomic selection model.
    
    Parameters:
    -----------
    pop : Pop
        Population to predict
    gsModel : dict
        Genomic selection model from RRBLUP
    simParam : SimParam
        Simulation parameters
    snpChip : int
        Which SNP chip to use
    
    Returns:
    --------
    Pop
        Population with EBV set
    """
    # Get SNP genotypes for prediction population
    M_pred = pullSnpGeno(pop, simParam, snpChip)
    
    # Center using training population means
    M_pred_centered = M_pred - gsModel['M_mean']
    
    # Calculate genomic relationship between training and prediction
    # G_pred_train = M_pred_centered @ M_train' / p
    n_markers = gsModel['n_markers']
    G_pred_train = np.dot(M_pred_centered, gsModel['M_train'].T) / n_markers
    
    # Calculate G_train for solving
    G_train = np.dot(gsModel['M_train'], gsModel['M_train'].T) / n_markers
    G_train += np.eye(G_train.shape[0]) * 1e-6
    
    # Solve: (G_train + lambda*I) * u = y
    A_train = G_train + gsModel['lambda'] * np.eye(G_train.shape[0])
    u_train = solve(A_train, gsModel['y_train'], assume_a='pos')
    
    # Predict: u_pred = G_pred_train @ u_train
    u_pred = np.dot(G_pred_train, u_train)
    
    # Set EBV in population
    if pop.ebv.shape[1] == 0:
        # Initialize EBV matrix if empty
        pop.ebv = np.zeros((pop.n_ind, 1))
    
    # Ensure EBV matrix has enough columns
    trait_idx = gsModel['trait'] - 1
    while pop.ebv.shape[1] <= trait_idx:
        pop.ebv = np.hstack([pop.ebv, np.zeros((pop.n_ind, 1))])
    
    pop.ebv[:, trait_idx] = u_pred
    
    return pop

## Global Parameters

Set up the simulation parameters for the clonal breeding program.

In [ ]:
# Number of simulation replications and breeding cycles
n_reps = 1  # Number of simulation replicates
n_burnin = 40  # Number of years in burnin phase
n_future = 40  # Number of years in future phase
start_records = 35  # Year when training and pedigree record collecting begins
n_cycles = n_burnin + n_future

# Genome simulation
n_chr = 15  # Number of chromosomes
n_qtl = 160  # Number of QTL per chromosome: 15 chr x 160 QTL = 2400 QTLs
n_snp = 600  # Simulate SNP chip with 9000 markers
gen_len = 1  # Genetic length
phy_len = 1e8  # Physical length
mut_rate = 2.5e-8  # Mutation rate

# Initial parents mean and variance
init_mean_g = 2500  # Phenotypic mean
init_var_g = 150000  # Genetic variance
init_var_ge = 150000  # Genotype-by-year interaction variance
var_e = 2800000  # Single variance

# Breeding program details
n_parents = 20  # Number of parents (and founders)
n_crosses = 100  # Number of crosses
n_progeny = 20  # Number of progenies per cross
n_clones_act = 500  # Number of individuals selected at ACT stage
n_clones_ect = 40  # Number of individuals selected at ECT stage

# Effective replication of yield trials
rep_hpt = 1  # h2 = 0.05
rep_act = 15  # h2 = 0.45
rep_ect = 50  # h2 = 0.65

scenario_name = "ClonalGS"

print(f"Simulation Parameters:")
print(f"  Replicates: {n_reps}")
print(f"  Burn-in years: {n_burnin}")
print(f"  Future years: {n_future}")
print(f"  Total cycles: {n_cycles}")
print(f"  Start collecting records: Year {start_records}")
print(f"  Chromosomes: {n_chr}")
print(f"  QTL per chromosome: {n_qtl}")
print(f"  SNP per chromosome: {n_snp}")
print(f"  Parents: {n_parents}")
print(f"  Crosses per year: {n_crosses}")
print(f"  Progeny per cross: {n_progeny}")

## Create Founders

Generate the initial founder population with haplotypes and set up simulation parameters.

In [ ]:
print("Creating founders...")

# Create founder population
founder_pop = runMacs2(
    nInd=n_parents,
    nChr=n_chr,
    segSites=n_qtl + n_snp,
    genLen=gen_len,
    mutRate=mut_rate
)

print(f"✓ Created founder population: {founder_pop.n_ind} individuals")

# Set simulation parameters
SP = SimParam(founder_pop)

# Restrict segregating sites (separate QTL and SNP)
SP.restrSegSites(minQtlPerChr=n_qtl, minSnpPerChr=n_snp)

# Add SNP chip
if n_snp > 0:
    SP.addSnpChip(n_snp)
    print(f"✓ Added SNP chip: {SP.n_snp_chips} SNP chips")

# Add traits: trait represents yield
# Using addTraitADG for additive, dominance, and GxE effects
SP.addTraitADG(
    nQtlPerChr=n_qtl,
    mean=init_mean_g,
    var=init_var_g,
    varGxE=init_var_ge
)
print(f"✓ Added TraitADG: {SP.n_traits} traits")

# Collect pedigree
SP.setTrackPed(True)
print("✓ Enabled pedigree tracking")

# Create founder parents
Parents = newPop(founder_pop, sim_param=SP)
print(f"✓ Created founder parents: {Parents.n_ind} individuals, {Parents.n_traits} traits")

# Set a phenotype to founder parents
Parents = setPheno(Parents, varE=var_e, reps=rep_ect, simParam=SP)

print(f"\nFounder population summary:")
print(f"  Mean genetic value: {meanG(Parents)[0]:.3f}")
print(f"  Genetic variance: {varG(Parents)[0]:.3f}")

## Fill Breeding Pipeline

Set up the initial breeding pipeline with 16 stages representing different evaluation years.
The pipeline includes:
- Stage 1: Crossing block (F1)
- Stages 2-4: Seedling evaluation (HPT1-3)
- Stages 5-9: Advanced clonal trials (ACT1-5)
- Stages 10-15: Elite clonal trials (ECT1-6)

**Note**: Year effects (p parameter) are not yet supported in AlphaSimPy's `setPheno` function.
The GxE variance is still included in the trait definition, which affects genetic values.

In [ ]:
print("Filling breeding pipeline...")

# Set initial yield trials with unique individuals
# Sample year effects
P = np.random.uniform(size=16)

# Breeding program
for cohort in range(1, 17):
    print(f"  FillPipeline stage: {cohort} of 16")
    
    # Stage 1: Crossing block
    F1 = randCross(Parents, nCrosses=n_crosses, nProgeny=n_progeny, simParam=SP)
    
    if cohort < 16:
        # Stage 2: Germinate the seedlings in the nursery
        Seedlings = setPheno(F1, varE=var_e, reps=rep_hpt, simParam=SP)
    
    if cohort < 15:
        # Stage 3: Plant in the seedlings in the field as HPT and record yields
        HPT1 = Seedlings
    
    if cohort < 14:
        # Stage 4: Record the HPT yields
        HPT2 = HPT1
    
    if cohort < 13:
        # Stage 5: Record the HPT yields
        HPT3 = setPheno(HPT2, varE=var_e, reps=rep_hpt, simParam=SP)
    
    if cohort < 12:
        # Stage 6: Select 500 superior individuals and plant as advanced clonal trials (ACT)
        ACT1 = selectInd(HPT3, nInd=n_clones_act, use="pheno", simParam=SP)
    
    if cohort < 11:
        # Stage 7: Record ACT yields
        ACT2 = ACT1
    
    if cohort < 10:
        # Stage 8: Record ACT yields
        ACT3 = ACT2
    
    if cohort < 9:
        # Stage 9: Record ACT yields
        ACT4 = ACT3
    
    if cohort < 8:
        # Stage 10: Record ACT yields
        ACT5 = setPheno(ACT4, varE=var_e, reps=rep_act, simParam=SP)
    
    if cohort < 7:
        # Stage 11: Select 40 superior individuals and plant as elite clonal trials (ECT)
        ECT1 = selectInd(ACT5, nInd=n_clones_ect, use="pheno", simParam=SP)
    
    if cohort < 6:
        # Stage 12: Record ECT yields
        ECT2 = ECT1
    
    if cohort < 5:
        # Stage 13: Record ECT yields
        ECT3 = ECT2
    
    if cohort < 4:
        # Stage 14: Record ECT yields
        ECT4 = ECT3
    
    if cohort < 3:
        # Stage 15: Record ECT yields
        ECT5 = ECT4
    
    if cohort < 2:
        # Stage 16: Record ECT yields
        ECT6 = setPheno(ECT5, varE=var_e, reps=rep_ect, simParam=SP)

print("\nPipeline filled successfully!")

## Main Simulation Loop

Run the breeding program simulation with burn-in (phenotypic selection) and future (genomic selection) phases.
In the future phase, HPT1-3 stages are replaced with genomic prediction.

In [ ]:
# Create list to store results from reps
results = []

for REP in range(1, n_reps + 1):
    print(f"\n{'='*60}")
    print(f"Working on REP: {REP}")
    print(f"{'='*60}")
    
    # Create a data frame to track key parameters
    output = {
        'year': list(range(1, n_cycles + 1)),
        'rep': [REP] * n_cycles,
        'scenario': [scenario_name] * n_cycles,
        'meanG': [0.0] * n_cycles,
        'varG': [0.0] * n_cycles,
        'accSel': [0.0] * n_cycles
    }
    
    # Initialize training population
    TrainPop = None
    
    # Simulate year effects
    P = np.random.uniform(size=n_cycles)
    
    # ---- Burn-in phase: Phenotypic selection program ----
    print("\n--> Working on Burn-in Phase (Phenotypic Selection)")
    for year in range(1, n_burnin + 1):
        print(f"  Working on burnin year: {year}")
        
        # Update parents (pick new parents)
        Parents = selectInd(ECT6, nInd=n_parents, use="pheno", simParam=SP)
        
        # Advance year (advances yield trials by a year and collects records)
        # Stage 16
        ECT6 = setPheno(ECT5, varE=var_e, reps=rep_ect, simParam=SP)
        
        # Stage 15
        ECT5 = ECT4
        
        # Stage 14
        ECT4 = ECT3
        
        # Stage 13
        ECT3 = ECT2
        
        # Stage 12
        ECT2 = ECT1
        
        # Stage 11
        ECT1 = selectInd(ACT5, nInd=n_clones_ect, use="pheno", simParam=SP)
        
        # Stage 10
        ACT5 = setPheno(ACT4, varE=var_e, reps=rep_act, simParam=SP)
        
        # Stage 9
        ACT4 = ACT3
        
        # Stage 8
        ACT3 = ACT2
        
        # Stage 7
        ACT2 = ACT1
        
        # Stage 6
        # Calculate accuracy based on 2000 inds (n_crosses * n_progeny)
        if HPT3.n_ind > 0:
            acc_sel = np.corrcoef(HPT3.gv[:, 0], HPT3.pheno[:, 0])[0, 1]
            output['accSel'][year-1] = acc_sel if not np.isnan(acc_sel) else 0.0
        ACT1 = selectInd(HPT3, nInd=n_clones_act, use="pheno", simParam=SP)
        
        # Stage 5
        HPT3 = setPheno(HPT2, varE=var_e, reps=rep_hpt, simParam=SP)
        
        # Stage 4
        HPT2 = HPT1
        
        # Stage 3
        HPT1 = Seedlings
        
        # Stage 2
        Seedlings = setPheno(F1, varE=var_e, reps=rep_hpt, simParam=SP)
        
        # Stage 1: Crossing block
        F1 = randCross(Parents, nCrosses=n_crosses, nProgeny=n_progeny, simParam=SP)
        
        # Store training population (starting from start_records)
        if year == start_records:
            print(f"    Start collecting training population (Year {year})")
            # Set fixEff for tracking year
            ACT5.fix_eff = [year] * ACT5.n_ind
            ECT6.fix_eff = [year] * ECT6.n_ind
            TrainPop = mergePops([ECT6])
        elif year > start_records and year < n_burnin + 1:
            print(f"    Collecting training population (Year {year})")
            # Set fixEff for tracking year
            ACT5.fix_eff = [year] * ACT5.n_ind
            ECT6.fix_eff = [year] * ECT6.n_ind
            TrainPop = mergePops([TrainPop, ECT6])
        
        # Report results
        output['meanG'][year-1] = meanG(Seedlings)[0]
        output['varG'][year-1] = varG(Seedlings)[0]
    
    # ---- Future phase: Genomic selection program ----
    # Replace three early stages (HPT1, HPT2, HPT3) with genomic prediction
    print("\n--> Working on Future Phase (Genomic Selection)")
    print("    HPT1-3 stages replaced with genomic prediction")
    
    for year in range(n_burnin + 1, n_burnin + n_future + 1):
        print(f"  Working on future year: {year}")
        
        # Run genomic model
        print("    Running GS model")
        gsModel = RRBLUP(TrainPop, SP, traits=1, use="pheno", snpChip=1, useReps=True)
        
        # Update parents (pick new parents)
        Parents = selectInd(ECT6, nInd=n_parents, use="pheno", simParam=SP)
        
        # Advance year (advances yield trials by a year and collects records)
        # Stage 13 (ECT6)
        ECT6 = setPheno(ECT5, varE=var_e, reps=rep_ect, simParam=SP)
        
        # Stage 12 (ECT5)
        ECT5 = ECT4
        
        # Stage 11 (ECT4)
        ECT4 = ECT3
        
        # Stage 10 (ECT3)
        ECT3 = ECT2
        
        # Stage 9 (ECT2)
        ECT2 = ECT1
        
        # Stage 8 (ECT1)
        ECT1 = selectInd(ACT5, nInd=n_clones_ect, use="pheno", simParam=SP)
        
        # Stage 7 (ACT5)
        ACT5 = setPheno(ACT4, varE=var_e, reps=rep_act, simParam=SP)
        
        # Stage 6 (ACT4)
        ACT4 = ACT3
        
        # Stage 5 (ACT3)
        ACT3 = ACT2
        
        # Stage 4 (ACT2)
        ACT2 = ACT1
        
        # Stage 3 (ACT1) - Using genomic selection instead of HPT stages
        # Calculate EBVs for Seedlings
        Seedlings = setEBV(Seedlings, gsModel, SP, snpChip=1)
        
        # Calculate accuracy based on 800 inds (ACT1 after GS selection)
        if Seedlings.n_ind > 0:
            acc_sel = np.corrcoef(Seedlings.gv[:, 0], Seedlings.ebv[:, 0])[0, 1]
            output['accSel'][year-1] = acc_sel if not np.isnan(acc_sel) else 0.0
        
        ACT1 = selectInd(Seedlings, nInd=n_clones_act, use="ebv", simParam=SP)
        
        # Stage 2 (Seedlings) - No HPT stages, go directly from F1
        Seedlings = setPheno(F1, varE=var_e, reps=rep_hpt, simParam=SP)
        
        # Stage 1: Crossing block
        F1 = randCross(Parents, nCrosses=n_crosses, nProgeny=n_progeny, simParam=SP)
        
        # Update training population (set to keep 6 years worth of records)
        print("    Maintaining training population")
        # Set fixEff for tracking year
        ACT5.fix_eff = [year] * ACT5.n_ind
        ECT6.fix_eff = [year] * ECT6.n_ind
        
        # Remove oldest records (one year's worth)
        if TrainPop.n_ind > 0:
            # Find oldest year
            if len(TrainPop.fix_eff) > 0:
                oldest_year = min(TrainPop.fix_eff)
                # Count individuals from oldest year
                n_remove = sum(1 for y in TrainPop.fix_eff if y == oldest_year)
                # Remove oldest records
                if n_remove > 0 and TrainPop.n_ind > n_remove:
                    keep_indices = [i for i in range(TrainPop.n_ind) 
                                  if TrainPop.fix_eff[i] != oldest_year]
                    TrainPop = subsetPop(TrainPop, keep_indices)
        
        # Add new records
        TrainPop = mergePops([TrainPop, ECT6])
        
        # Report results
        output['meanG'][year-1] = meanG(Seedlings)[0]
        output['varG'][year-1] = varG(Seedlings)[0]
    
    # Save results from current replicate
    results.append(output)

print("\n" + "="*60)
print("Simulation completed!")
print("="*60)

## Analyze Results

Visualize the results from the simulation.

In [ ]:
# Combine results from all replicates
import pandas as pd

df = pd.DataFrame(results[0])  # For single replicate, convert dict to DataFrame

# If multiple replicates, combine them
if len(results) > 1:
    df = pd.concat([pd.DataFrame(r) for r in results], ignore_index=True)

print("Results summary:")
print(df.head(10))
print(f"\nTotal years simulated: {len(df)}")

In [ ]:
# Plotting function
def plot_results(x, y, title, xlabel, ylabel, ylim=None):
    plt.plot(x, y, 'b-', linewidth=2)
    plt.axvline(x=n_burnin, color='r', linestyle='--', label='GS Start')
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    if ylim is not None:
        plt.ylim(ylim)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()

# Create plots
fig, axes = plt.subplots(3, 1, figsize=(6, 12))

# Genetic Gain
plt.sca(axes[0])
plot_results(df['year'], df['meanG'], 
              'Genetic gain', 'Year', 'Yield')

# Genetic Variance
plt.sca(axes[1])
plot_results(df['year'], df['varG'], 
              'Genetic variance', 'Year', 'Variance')

# Selection Accuracy
plt.sca(axes[2])
plot_results(df['year'], df['accSel'], 
              'Selection accuracy', 'Year', 'Correlation')

plt.tight_layout()
plt.savefig('GenomicSelection_Results.png', dpi=150, bbox_inches='tight')
plt.show()

print("Results plot saved as 'GenomicSelection_Results.png'")

## Summary

This tutorial demonstrated:

1. **Founder Population Creation**: Using `runMacs2` to generate initial haplotypes for clonal species
2. **Trait Definition**: Adding traits with additive, dominance, and GxE effects using `addTraitADG`
3. **Breeding Pipeline**: Setting up a 16-stage clonal breeding pipeline with:
   - Seedling evaluation (HPT stages) - replaced with GS in future phase
   - Advanced clonal trials (ACT stages)
   - Elite clonal trials (ECT stages)
4. **Phenotypic Selection (Burn-in)**: Selecting superior clones at each stage based on phenotypic performance
5. **Genomic Selection (Future)**: Replacing HPT1-3 stages with genomic prediction using RRBLUP:
   - Training population collected from ECT6 and ACT5 stages
   - Genomic predictions used to select superior seedlings before ACT stage
   - Accelerates breeding cycle by eliminating early-stage phenotyping
6. **Training Population Management**: Maintaining a rolling window of training data (6 years)
7. **Genetic Progress**: Tracking genetic gain, variance, and selection accuracy over time

The simulation shows how genomic selection can accelerate clonal breeding programs by replacing 
early-stage phenotypic evaluation with genomic prediction, reducing time and resources while 
maintaining or improving genetic gain.